In [1]:
# 02_feature_engineering.ipynb

# 📦 Imports
import pandas as pd
import numpy as np
from datetime import date


In [2]:
# 🔧 Config

# Leagues:
# Premier-League - 9
# La-Liga - 12
# Serie-A - 11
# Ligue-1
# Bundesliga - 20

league = 'Premier-League'
season = "2024-2025"  # change to 2024-2025 when needed
filename = f"sot_consolidated_{league}_{season}.csv"
rolling_window = 3  # number of previous matches to use in rolling averages
min_gameweek_for_features = 4  # earliest week with reliable rolling features
export = True
processed_data_path = '/Users/tylermartins/Documents/PitchSights/ps-betting/strategies/shots_on_target/data/processed'

In [3]:
filename = f"{processed_data_path}/sot_consolidated_{league}_{season}.csv"
final_df = pd.read_csv(filename)
final_df = final_df.drop(columns=[col for col in final_df.columns if "Unnamed" in col])


In [4]:
# 🗓️ Date and Gameweek
final_df["match_date"] = pd.to_datetime(final_df["match_date"])
final_df = final_df.sort_values(["team", "player", "match_date"]).reset_index(drop=True)
final_df["gameweek_number"] = final_df.groupby("team")["match_date"].rank(method="dense").astype(int)

In [5]:
final_df[(final_df['team'] == 'Southampton')&(final_df['match_date'] == '2024-08-24')]

,match_date,player,team,position,age_decimal,minutes,goals,shots,shots_on_target,xg,...,opp_possession,team_passing_accuracy,opp_passing_accuracy,opp_shots_on_target,opp_shots,team_saves,opp_saves,team_saves_faced,opp_saves_faced,gameweek_number
26820,2024-08-24,Flynn Downes,Southampton,CM,25.59,90,0,0,0,0.0,...,36,89,80,8,23,7,1,8,1,2
26821,2024-08-24,Flynn Downes,Southampton,CM,25.59,90,0,0,0,0.0,...,36,89,80,8,23,7,1,8,1,2


In [6]:
player_match_unique = final_df.copy()
player_match_unique = player_match_unique.drop(columns=['line','bookmaker','bet_odds','closing_odds']).drop_duplicates()

player_match_lines = final_df.copy()
player_match_lines = player_match_lines[['match_date','player','line','bookmaker','bet_odds','closing_odds']]

player_match_unique[player_match_unique['player'].str.contains('Cunha')]


,match_date,player,team,position,age_decimal,minutes,goals,shots,shots_on_target,xg,...,opp_possession,team_passing_accuracy,opp_passing_accuracy,opp_shots_on_target,opp_shots,team_saves,opp_saves,team_saves_faced,opp_saves_faced,gameweek_number
31005,2024-08-17,Matheus Cunha,Wolves,FW,25.22,34,0,1,1,0.1,...,53,80,83,6,18,4,3,6,3,1
31010,2024-08-25,Matheus Cunha,Wolves,AM,25.25,67,1,4,2,0.9,...,60,76,85,8,14,2,2,8,4,2
31016,2024-08-31,Matheus Cunha,Wolves,FW,25.26,90,0,4,0,0.2,...,52,76,76,5,16,4,1,5,2,3
31021,2024-09-15,Matheus Cunha,Wolves,"RM,FW",25.30,90,0,4,2,0.2,...,51,81,86,6,14,4,4,6,5,4
31030,2024-09-21,Matheus Cunha,Wolves,LM,25.32,90,1,1,1,0.0,...,53,84,86,3,9,0,3,3,4,5
31036,2024-09-28,Matheus Cunha,Wolves,"RM,LM",25.34,90,0,2,1,0.0,...,55,82,83,5,9,4,2,5,3,6
31047,2024-10-05,Matheus Cunha,Wolves,LM,25.36,90,1,5,1,0.4,...,44,77,75,11,18,7,3,11,6,7
31056,2024-10-20,Matheus Cunha,Wolves,FW,25.40,66,0,1,0,0.0,...,77,67,86,7,22,5,1,7,2,8
31064,2024-10-26,Matheus Cunha,Wolves,"FW,AM",25.42,90,1,5,2,0.2,...,51,81,83,6,19,4,5,6,7,9
31072,2024-11-02,Matheus Cunha,Wolves,AM,25.44,90,0,2,0,0.1,...,43,79,74,6,19,4,4,6,6,10


In [7]:
# ⚽ Rolling totals for raw stats and minutes
raw_fields = [
    "shots_on_target","goals", "shots", "xg", "npxg", "xg_assist", "sca", "gca",
    "touches_att_3rd", "touches_att_pen_area", "miscontrols", "dispossessed",
    "passes_received", "progressive_passes_received", "take_ons_won", "take_ons_tackled", "minutes"
]


# Rolling sums per player per raw field
for col in raw_fields:
    player_match_unique[f"{col}_ma"] = (
        player_match_unique.groupby(["player"])[col]
        .transform(lambda x: x.shift().rolling(rolling_window, min_periods=1).mean())
    )

In [8]:
# ✅ Rolling per90 stats based on rolling totals
def safe_per90(numerator, minutes):
    return numerator / (minutes / 90 + 1e-6)

In [9]:
player_match_unique["goals_per90_ma"] = safe_per90(player_match_unique["goals_ma"], player_match_unique["minutes_ma"])
player_match_unique["sot_per90_ma"] = safe_per90(player_match_unique["shots_on_target_ma"], player_match_unique["minutes_ma"])
player_match_unique["shots_per90_ma"] = safe_per90(player_match_unique["shots_ma"], player_match_unique["minutes_ma"])
player_match_unique["xg_per90_ma"] = safe_per90(player_match_unique["xg_ma"], player_match_unique["minutes_ma"])
player_match_unique["npxg_per90_ma"] = safe_per90(player_match_unique["npxg_ma"], player_match_unique["minutes_ma"])
player_match_unique["xa_per90_ma"] = safe_per90(player_match_unique["xg_assist_ma"], player_match_unique["minutes_ma"])
player_match_unique["sca_per90_ma"] = safe_per90(player_match_unique["sca_ma"], player_match_unique["minutes_ma"])
player_match_unique["gca_per90_ma"] = safe_per90(player_match_unique["gca_ma"], player_match_unique["minutes_ma"])
player_match_unique["touches_att_third_per90_ma"] = safe_per90(player_match_unique["touches_att_3rd_ma"], player_match_unique["minutes_ma"])
player_match_unique["touches_att_pen_area_per90_ma"] = safe_per90(player_match_unique["touches_att_pen_area_ma"], player_match_unique["minutes_ma"])
player_match_unique["miscontrols_per90_ma"] = safe_per90(player_match_unique["miscontrols_ma"], player_match_unique["minutes_ma"])
player_match_unique["dispossessed_per90_ma"] = safe_per90(player_match_unique["dispossessed_ma"], player_match_unique["minutes_ma"])
player_match_unique["passes_received_per90_ma"] = safe_per90(player_match_unique["passes_received_ma"], player_match_unique["minutes_ma"])
player_match_unique["progressive_passes_received_per90_ma"] = safe_per90(player_match_unique["progressive_passes_received_ma"], player_match_unique["minutes_ma"])
player_match_unique["take_ons_won_per90_ma"] = safe_per90(player_match_unique["take_ons_won_ma"], player_match_unique["minutes_ma"])
player_match_unique["take_ons_tackled_per90_ma"] = safe_per90(player_match_unique["take_ons_tackled_ma"], player_match_unique["minutes_ma"])

# 📊 Team-Level Rolling Stats
team_match_unique =  player_match_unique.copy()

# ensure types
team_match_unique = team_match_unique.sort_values(["team", "match_date"])
# keep only the columns we need for dedup + team stats
keep_cols = [
    "match_date", "team",
    "team_shots", "team_shots_on_target", "team_xG",
    "player", "bookmaker", "line", f"bet_odds"
]
keep_cols = [c for c in keep_cols if c in team_match_unique.columns]  # guard
team_match_unique = team_match_unique[keep_cols].dropna(subset=["match_date", "team"])

# 2) dedup by player-line-bookmaker-odds first, then reduce to unique team-date
team_match_unique = (
    team_match_unique
    .sort_values(["team","match_date"])  # ensures rolling is correct
    .groupby(["match_date","team"], as_index=False)
    .first()  # keep the first occurrence of each team in each match
)

# 3) compute rolling team MAs on the deduped frame
for col in ["team_shots", "team_shots_on_target", "team_xG"]:
    if col in team_match_unique.columns:
        team_match_unique[f"{col}_ma"] = (
            team_match_unique
            .groupby("team")[col]
            .transform(lambda x: x.shift().rolling(rolling_window, min_periods=1).mean())
        )

# keep only the MA columns to merge back
team_ma_cols = ["match_date","team"] + [c for c in team_match_unique.columns if c.endswith("_ma")]
team_match_unique = team_match_unique[team_ma_cols].copy()

# 4) merge MAs back onto player_match_unique
player_match_unique = pd.merge(player_match_unique, team_match_unique, on=["match_date","team"], how="left")

# 📈 Share Features
player_match_unique["sot_team_share"] = player_match_unique["sot_per90_ma"] / (player_match_unique["team_shots_on_target_ma"] + 0.01)
player_match_unique["shots_team_share"] = player_match_unique["shots_per90_ma"] / (player_match_unique["team_shots_ma"] + 0.01)
player_match_unique["xg_team_share"] = player_match_unique["xg_per90_ma"] / (player_match_unique["team_xG_ma"] + 0.01)




In [10]:
final_df = pd.merge(player_match_lines,player_match_unique, how = 'inner', on=['match_date','player'])

In [11]:
# 📉 Market Sentiment
final_df["implied_prob"] = 1 / final_df["bet_odds"]

# ⚙️ Contextual Features
final_df["is_home"] = (final_df["team"] == final_df["home_team"]).astype(int)
final_df["line_to_sot_ratio"] = final_df["line"] / (final_df["sot_per90_ma"] + 0.01)
final_df["opponent_sot_allowed_per90"] = final_df["opp_shots_on_target"] / 90
final_df["opponent_xg_allowed_per90"] = final_df["opp_xG"] / 90

In [12]:
# 🚦 Model Confidence (simple GW-based logic for now)
final_df = final_df[final_df["gameweek_number"] >= min_gameweek_for_features].copy()
final_df["model_confidence"] = np.where(
    final_df["gameweek_number"] <= min_gameweek_for_features + 1, "low", "high"
)


In [13]:
# 🎯 Target Variable
final_df["target_hit_line"] = (final_df["shots_on_target"] >= final_df["line"]).astype(int)




In [14]:
final_df.columns

Index(['match_date', 'player', 'line', 'bookmaker', 'bet_odds', 'closing_odds',
       'team', 'position', 'age_decimal', 'minutes', 'goals', 'shots',
       'shots_on_target', 'xg', 'npxg', 'xg_assist', 'sca', 'gca',
       'miscontrols', 'dispossessed', 'passes_received',
       'progressive_passes_received', 'take_ons', 'take_ons_won',
       'take_ons_tackled', 'touches_att_3rd', 'touches_att_pen_area',
       'team_shots', 'team_shots_on_target', 'team_xG', 'home_team',
       'away_team', 'opp', 'opp_xG', 'team_score', 'opp_score',
       'team_possession', 'opp_possession', 'team_passing_accuracy',
       'opp_passing_accuracy', 'opp_shots_on_target', 'opp_shots',
       'team_saves', 'opp_saves', 'team_saves_faced', 'opp_saves_faced',
       'gameweek_number', 'shots_on_target_ma', 'goals_ma', 'shots_ma',
       'xg_ma', 'npxg_ma', 'xg_assist_ma', 'sca_ma', 'gca_ma',
       'touches_att_3rd_ma', 'touches_att_pen_area_ma', 'miscontrols_ma',
       'dispossessed_ma', 'passes_rece

In [15]:
# Step 1 is handled in the model script (target variable usage)
# Step 2 - Drop under odds columns and any rows with missing data
final_df_cleaned = final_df

# Step 3 - Convert match_date to datetime and sort chronologically
final_df_cleaned['match_date'] = pd.to_datetime(final_df_cleaned['match_date'])
final_df_cleaned = final_df_cleaned.sort_values(by=['match_date', 'gameweek_number'])

# Step 4 - Drop columns that should not be used as features
columns_to_drop = [
    'match_date', 'player', 'line', 'bookmaker', 'bet_odds','closing_odds',
       'team', 'position', 'age_decimal', 'minutes', 'goals', 'shots',
       'shots_on_target', 'xg', 'npxg', 'xg_assist', 'sca', 'gca',
       'miscontrols', 'dispossessed', 'passes_received',
       'progressive_passes_received', 'take_ons', 'take_ons_won',
       'take_ons_tackled', 'touches_att_3rd', 'touches_att_pen_area',
       'team_shots', 'team_shots_on_target', 'team_xG', 'home_team',
       'away_team', 'opp', 'opp_xG', 'team_score', 'opp_score',
       'team_possession', 'opp_possession', 'team_passing_accuracy',
       'opp_passing_accuracy', 'opp_shots_on_target', 'opp_shots',
       'team_saves', 'opp_saves', 'team_saves_faced', 'opp_saves_faced',
       'gameweek_number', 'shots_on_target_ma', 'goals_ma', 'shots_ma',
       'xg_ma', 'npxg_ma', 'xg_assist_ma', 'sca_ma', 'gca_ma',
       'touches_att_3rd_ma', 'touches_att_pen_area_ma', 'miscontrols_ma',
       'dispossessed_ma', 'passes_received_ma',
       'progressive_passes_received_ma', 'take_ons_won_ma',
       'take_ons_tackled_ma','model_confidence',
    'target_hit_line'  # this is the label, will be used in the model script
]

feature_cols = [col for col in final_df_cleaned.columns if col not in columns_to_drop]

# Save the cleaned dataset and feature list
# cleaned_path = "/mnt/data/features_cleaned_for_model.csv"
# features_list_path = "/mnt/data/feature_columns.txt"
# final_df_cleaned.to_csv(cleaned_path, index=False)

# with open(features_list_path, 'w') as f:
#     f.write('\n'.join(feature_cols))

# (cleaned_path, features_list_path)


In [16]:
final_df_cleaned

,match_date,player,line,bookmaker,bet_odds,closing_odds,team,position,age_decimal,minutes,...,sot_team_share,shots_team_share,xg_team_share,implied_prob,is_home,line_to_sot_ratio,opponent_sot_allowed_per90,opponent_xg_allowed_per90,model_confidence,target_hit_line
2006,2024-09-14,Amadou Onana,0.5,BetRivers,3.15,3.05,Aston Villa,DM,23.08,45,...,NaN,NaN,NaN,0.317460,1,NaN,0.022222,0.010000,low,0
2007,2024-09-14,Amadou Onana,0.5,DraftKings,2.50,2.50,Aston Villa,DM,23.08,45,...,NaN,NaN,NaN,0.400000,1,NaN,0.022222,0.010000,low,0
2008,2024-09-14,Amadou Onana,1.0,FanDuel,2.60,2.60,Aston Villa,DM,23.08,45,...,NaN,NaN,NaN,0.384615,1,NaN,0.022222,0.010000,low,0
2009,2024-09-14,Amadou Onana,1.5,DraftKings,11.00,11.00,Aston Villa,DM,23.08,45,...,NaN,NaN,NaN,0.090909,1,NaN,0.022222,0.010000,low,0
2010,2024-09-14,Amadou Onana,2.0,FanDuel,12.00,12.00,Aston Villa,DM,23.08,45,...,NaN,NaN,NaN,0.083333,1,NaN,0.022222,0.010000,low,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31640,2025-05-25,Santiago Bueno,0.5,BetRivers,6.10,NaN,Wolves,"AM,CB",26.54,4,...,0.0,0.0,0.0,0.163934,1,50.0,0.077778,0.015556,high,0
31641,2025-05-25,Santiago Bueno,1.0,FanDuel,6.00,NaN,Wolves,"AM,CB",26.54,4,...,0.0,0.0,0.0,0.166667,1,100.0,0.077778,0.015556,high,0
31765,2025-05-25,Toti Gomes,0.5,BetRivers,5.40,5.40,Wolves,CB,26.35,90,...,0.0,0.0,0.0,0.185185,1,50.0,0.077778,0.015556,high,0
31766,2025-05-25,Toti Gomes,1.0,FanDuel,4.30,4.30,Wolves,CB,26.35,90,...,0.0,0.0,0.0,0.232558,1,100.0,0.077778,0.015556,high,0


In [17]:
if export:
    export_filename = f"{processed_data_path}/features_{league}_{season}.csv"
    final_df.to_csv(export_filename, index=False)

print("✅ Feature engineering complete.")


✅ Feature engineering complete.
